# Estimación Costes Stock: CatBoost vs Naive

Este notebook implementa el análisis de costes de inventario comparando:
- **Modelo CatBoost**: Predicciones del mejor modelo ML
- **Modelo Naive**: Media móvil de 7 días

Se utiliza la fórmula de Yamazaki (2015) para calcular el stock de seguridad:
$$SS = Z \times \sigma \times \sqrt{L}$$

Donde:
- $Z$: Factor de servicio (1.645 para 95%). Distribución normal estándar inversa para probailidad dada (%Nivel del servicio)
- $\sigma$: Desviación estándar del error (RMSE)
- $L$: Lead time (días de aprovisionamiento), Se toman los días entre pedidos más los días de lead time

**Referencia**: Yamazaki, Y. (2015). DOI: https://doi.org/10.1080/00207543.2015.1076179

# Importaciones

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

# Detectar si estamos en Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Configurar el directorio de trabajo según el entorno
if IN_COLAB:
    os.chdir('TFMDS')
else:
    # Detectar si estamos en Codespaces o VS Code local
    if os.path.exists('/workspaces/TFMDS'):
        # Entorno Codespaces
        os.chdir('/workspaces/TFMDS')
    else:
        # En VS Code local, nos movemos al directorio raíz del proyecto
        # Usa raw string para evitar errores de escape en rutas Windows
        current_dir = r'C:\Users\jmora\Documents\TFMDS'
        os.chdir(current_dir)

# OPCIONAL: Para verificar que estás en la ruta correcta y ver las carpetas
print("Directorio de trabajo actual:", os.getcwd())


Directorio de trabajo actual: C:\Users\jmora\Documents\TFMDS


# Parámetros

In [2]:

# Parámetros globales
TASA_ALMACENAMIENTO = 0.05    # 5% del valor del producto
NIVEL_SERVICIO = 0.95               # 95% de nivel de servicio
Z_SCORE = norm.ppf(NIVEL_SERVICIO)  # Factor Z para %NIVEL_SERVICIO5 de confianza ppf = inversa de la CDF normal estándar

print("Configuración cargada")
print(f"Nivel de servicio: {NIVEL_SERVICIO*100}%")
print(f"Z-score: {Z_SCORE}")
print(f"Tasa almacenamiento anual: {TASA_ALMACENAMIENTO*100}%")

Configuración cargada
Nivel de servicio: 95.0%
Z-score: 1.6448536269514722
Tasa almacenamiento anual: 5.0%


# Carga de Datos

In [3]:
# Cargar predicciones de CatBoost
df_test = pd.read_csv('datos/df_test_catboost.csv', sep=';', decimal=',')
print(f"Datos de test cargados: {df_test.shape}")
print(f"Columnas: {df_test.columns.tolist()}")

# Cargar datos de ciclo de aprovisionamiento
df_ciclos = pd.read_csv('datos/DatosCicloAprovisionamiento.csv', sep=';', decimal=',')
print(f"\nDatos de ciclos: {df_ciclos.shape}")
print(f"Columnas: {df_ciclos.columns.tolist()}")

# Cargar precios medios
df_precios = pd.read_csv('datos/DatosPrecioMedio.csv', sep=';', decimal=',')
# Convertir precio de formato europeo (coma) a float
if df_precios['eurPrecioMedio'].dtype == 'object':
    df_precios['eurPrecioMedio'] = df_precios['eurPrecioMedio'].str.replace(',', '.').astype(float)
print(f"\nDatos de precios: {df_precios.shape}")
print(f"Columnas: {df_precios.columns.tolist()}")

# Vista previa
print("\nPrimeras filas de df_test:")
display(df_test.head())
print("\nPrimeras filas de df_ciclos:")
display(df_ciclos.head())
print("\nPrimeras filas de df_precios:")
display(df_precios.head())

Datos de test cargados: (23244, 4)
Columnas: ['idSecuencia', 'producto', 'udsVenta', 'udsVentaPred']

Datos de ciclos: (1000, 3)
Columnas: ['producto', 'diasEntrePedidos', 'diasLeadtime']

Datos de precios: (1001, 2)
Columnas: ['producto', 'eurPrecioMedio']

Primeras filas de df_test:


,idSecuencia,producto,udsVenta,udsVentaPred
0,2024-10-07,1,2,7.909209
1,2024-10-08,1,0,7.026752
2,2024-10-09,1,5,5.903247
3,2024-10-10,1,49,6.543139
4,2024-10-11,1,7,11.997252



Primeras filas de df_ciclos:


,producto,diasEntrePedidos,diasLeadtime
0,1,14,15
1,2,14,15
2,3,14,15
3,4,14,15
4,5,14,15



Primeras filas de df_precios:


,producto,eurPrecioMedio
0,1.0,68.730000
1,2.0,148.330000
2,3.0,169.000000
3,4.0,0.604383
4,5.0,4.553314


# Preparación de Datos

In [4]:
# Preparar el dataframe con todas las columnas necesarias
# Convertir idSecuencia a datetime
df_test['fecha'] = pd.to_datetime(df_test['idSecuencia'])

# Merge con ciclos de aprovisionamiento
df_test = df_test.merge(df_ciclos[['producto', 'diasEntrePedidos', 'diasLeadtime']], 
                         on='producto', how='left')

# Calcular ciclo total de aprovisionamiento (L en la fórmula)
df_test['ciclo_aprovisionamiento'] = df_test['diasEntrePedidos'] + df_test['diasLeadtime']

# Merge con precios
df_test = df_test.merge(df_precios[['producto', 'eurPrecioMedio']], 
                         on='producto', how='left')

# Verificar datos faltantes
print("Datos faltantes por columna:")
print(df_test.isnull().sum())
print(f"\nTotal de registros: {len(df_test)}")
print(f"Productos únicos: {df_test['producto'].nunique()}")
print(f"Rango de fechas: {df_test['fecha'].min()} a {df_test['fecha'].max()}")

# Vista previa
display(df_test)

Datos faltantes por columna:
idSecuencia                0
producto                   0
udsVenta                   0
udsVentaPred               0
fecha                      0
diasEntrePedidos           0
diasLeadtime               0
ciclo_aprovisionamiento    0
eurPrecioMedio             0
dtype: int64

Total de registros: 23244
Productos únicos: 894
Rango de fechas: 2024-10-07 00:00:00 a 2024-11-05 00:00:00


,idSecuencia,producto,udsVenta,udsVentaPred,fecha,diasEntrePedidos,diasLeadtime,ciclo_aprovisionamiento,eurPrecioMedio
0,2024-10-07,1,2,7.909209,2024-10-07,14,15,29,68.73
1,2024-10-08,1,0,7.026752,2024-10-08,14,15,29,68.73
2,2024-10-09,1,5,5.903247,2024-10-09,14,15,29,68.73
3,2024-10-10,1,49,6.543139,2024-10-10,14,15,29,68.73
4,2024-10-11,1,7,11.997252,2024-10-11,14,15,29,68.73
...,...,...,...,...,...,...,...,...,...
23239,2024-10-31,1000,0,1.264586,2024-10-31,14,2,16,40.73
23240,2024-11-01,1000,0,0.190684,2024-11-01,14,2,16,40.73
23241,2024-11-02,1000,0,1.376287,2024-11-02,14,2,16,40.73
23242,2024-11-04,1000,7,1.311637,2024-11-04,14,2,16,40.73


# Modelo Naive: Media Móvil 7 días

Implementación el modelo baseline utilizando una media móvil de 7 días como predicción.

In [5]:
# Ordenar por producto y fecha para calcular media móvil
df_test = df_test.sort_values(['producto', 'fecha']).reset_index(drop=True)

# Crear predicción Naive: media móvil de 7 días
df_test['udsVentaPred_Naive'] = df_test.groupby('producto')['udsVenta'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean().shift(1)
)


# Para los primeros días sin histórico suficiente, usar la media general del producto
df_test['udsVentaPred_Naive'] = df_test.groupby('producto')['udsVentaPred_Naive'].transform(
    lambda x: x.fillna(x.mean())
)

print("Predicciones Naive calculadas")
print(f"Valores nulos en Naive: {df_test['udsVentaPred_Naive'].isnull().sum()}")

# Comparativa de predicciones
print("\nEstadísticas de predicciones:")
print(df_test[['udsVenta', 'udsVentaPred', 'udsVentaPred_Naive']].describe())

display(df_test[['fecha', 'producto', 'udsVenta', 'udsVentaPred', 'udsVentaPred_Naive']].head(20))

Predicciones Naive calculadas
Valores nulos en Naive: 0

Estadísticas de predicciones:
           udsVenta  udsVentaPred  udsVentaPred_Naive
count  23244.000000  23244.000000        23244.000000
mean       1.632851      1.615016            1.656210
std        2.768287      1.345529            1.860696
min        0.000000     -0.841746            0.000000
25%        0.000000      0.727191            0.285714
50%        0.000000      1.237118            1.000000
75%        2.000000      2.084822            2.285714
max       49.000000     15.883632           21.000000


,fecha,producto,udsVenta,udsVentaPred,udsVentaPred_Naive
0,2024-10-07,1,2,7.909209,11.407810
1,2024-10-08,1,0,7.026752,2.000000
2,2024-10-09,1,5,5.903247,1.000000
3,2024-10-10,1,49,6.543139,2.333333
4,2024-10-11,1,7,11.997252,14.000000
5,2024-10-12,1,2,2.545331,12.600000
6,2024-10-14,1,16,7.442095,10.833333
7,2024-10-15,1,5,8.324207,11.571429
8,2024-10-16,1,28,9.598146,12.000000
9,2024-10-17,1,35,14.392503,16.000000


# Cálculo del RMSE por Producto

Calculamos el RMSE (desviación estándar de la demanda) para cada producto en ambos modelos.

In [6]:
# Función para calcular RMSE
def calcular_rmse(y_true, y_pred):
    """Calcula el RMSE entre valores reales y predichos"""
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Calcular RMSE por producto para ambos modelos
rmse_por_producto = df_test.groupby('producto').apply(
    lambda x: pd.Series({
        'RMSE_CatBoost': calcular_rmse(x['udsVenta'], x['udsVentaPred']),
        'RMSE_Naive': calcular_rmse(x['udsVenta'], x['udsVentaPred_Naive']),
        'n_observaciones': len(x),
        'ciclo_aprovisionamiento': x['ciclo_aprovisionamiento'].iloc[0],
        'precio_medio': x['eurPrecioMedio'].iloc[0]
    })
).reset_index()

# Calcular la mejora del modelo CatBoost respecto a Naive
num = rmse_por_producto['RMSE_Naive'] - rmse_por_producto['RMSE_CatBoost']
den = rmse_por_producto['RMSE_Naive']
rmse_por_producto['mejora_RMSE_%'] = np.where(den == 0, 0, num / den * 100)

# Estadísticas globales
print("=" * 80)
print("ESTADÍSTICAS GLOBALES DE RMSE")
print("=" * 80)
print(f"\nRMSE promedio CatBoost: {rmse_por_producto['RMSE_CatBoost'].mean():.2f}")
print(f"RMSE promedio Naive: {rmse_por_producto['RMSE_Naive'].mean():.2f}")
print(f"Mejora promedio: {rmse_por_producto['mejora_RMSE_%'].mean():.2f}%")
print(f"\nProductos donde CatBoost es mejor: {(rmse_por_producto['mejora_RMSE_%'] > 0).sum()}/{len(rmse_por_producto)}")

# Mostrar tabla completa
print("\n" + "=" * 80)
print("RMSE POR PRODUCTO")
print("=" * 80)
display(rmse_por_producto.sort_values('mejora_RMSE_%', ascending=False))

ESTADÍSTICAS GLOBALES DE RMSE

RMSE promedio CatBoost: 2.03
RMSE promedio Naive: 2.14
Mejora promedio: 4.59%

Productos donde CatBoost es mejor: 691/894

RMSE POR PRODUCTO


,producto,RMSE_CatBoost,RMSE_Naive,n_observaciones,ciclo_aprovisionamiento,precio_medio,mejora_RMSE_%
506,573,1.520753,2.168640,26.0,16.0,97.210000,29.875257
380,430,1.161248,1.638553,26.0,16.0,17.613333,29.129669
325,364,3.142170,4.324357,26.0,16.0,70.380000,27.337878
220,243,1.177248,1.586237,26.0,16.0,17.880000,25.783597
111,126,1.227547,1.652878,26.0,16.0,22.600000,25.732771
...,...,...,...,...,...,...,...
297,328,1.835443,1.048819,26.0,16.0,7.396529,-75.000841
575,649,1.114107,0.635169,26.0,16.0,17.330000,-75.403404
514,582,0.759945,0.419607,26.0,16.0,5.253861,-81.108547
695,783,0.778565,0.396221,26.0,16.0,20.980000,-96.497864


# Cálculo del Stock de Seguridad

Aplicamos la fórmula de Yamazaki (2015):

$$SS = Z \times \sigma \times \sqrt{L}$$

Donde:
- $Z$ = Factor de servicio (ya calculado en parámetros)
- $\sigma$ = RMSE del modelo
- $L$ = Ciclo de aprovisionamiento (días)

In [7]:
# Añadir RMSE al dataframe principal mediante merge
df_test = df_test.merge(
    rmse_por_producto[['producto', 'RMSE_CatBoost', 'RMSE_Naive']], 
    on='producto', 
    how='left'
)

# Calcular stock de seguridad según fórmula de Yamazaki
# SS = Z * RMSE * sqrt(L)

df_test['stock_seguridad_CatBoost'] = (
    Z_SCORE * 
    df_test['RMSE_CatBoost'] * 
    np.sqrt(df_test['ciclo_aprovisionamiento'])
)

df_test['stock_seguridad_Naive'] = (
    Z_SCORE * 
    df_test['RMSE_Naive'] * 
    np.sqrt(df_test['ciclo_aprovisionamiento'])
)

# Calcular reducción de stock
df_test['reduccion_stock_unidades'] = (
    df_test['stock_seguridad_Naive'] - df_test['stock_seguridad_CatBoost']
)

df_test['reduccion_stock_%'] = np.where(
    df_test['stock_seguridad_Naive'] == 0,
    0,
    (df_test['reduccion_stock_unidades'] / df_test['stock_seguridad_Naive']) * 100
)

# Resumen
print("=" * 80)
print("STOCK DE SEGURIDAD - ESTADÍSTICAS")
print("=" * 80)
print(f"\nStock promedio CatBoost: {df_test['stock_seguridad_CatBoost'].mean():.2f} unidades")
print(f"Stock promedio Naive: {df_test['stock_seguridad_Naive'].mean():.2f} unidades")
print(f"Reducción promedio: {df_test['reduccion_stock_unidades'].mean():.2f} unidades ({df_test['reduccion_stock_%'].mean():.2f}%)")

# Mostrar ejemplos
print("\n" + "=" * 80)
print("EJEMPLOS DE STOCK DE SEGURIDAD")
print("=" * 80)
display(df_test[['fecha', 'producto', 'ciclo_aprovisionamiento', 'RMSE_CatBoost', 'RMSE_Naive',
                  'stock_seguridad_CatBoost', 'stock_seguridad_Naive', 'reduccion_stock_unidades']])

STOCK DE SEGURIDAD - ESTADÍSTICAS

Stock promedio CatBoost: 14.72 unidades
Stock promedio Naive: 15.56 unidades
Reducción promedio: 0.85 unidades (4.59%)

EJEMPLOS DE STOCK DE SEGURIDAD


,fecha,producto,ciclo_aprovisionamiento,RMSE_CatBoost,RMSE_Naive,stock_seguridad_CatBoost,stock_seguridad_Naive,reduccion_stock_unidades
0,2024-10-07,1,29,10.970975,12.297644,97.178791,108.930164,11.751373
1,2024-10-08,1,29,10.970975,12.297644,97.178791,108.930164,11.751373
2,2024-10-09,1,29,10.970975,12.297644,97.178791,108.930164,11.751373
3,2024-10-10,1,29,10.970975,12.297644,97.178791,108.930164,11.751373
4,2024-10-11,1,29,10.970975,12.297644,97.178791,108.930164,11.751373
...,...,...,...,...,...,...,...,...
23239,2024-10-31,1000,16,2.310619,2.584463,15.202518,17.004251,1.801733
23240,2024-11-01,1000,16,2.310619,2.584463,15.202518,17.004251,1.801733
23241,2024-11-02,1000,16,2.310619,2.584463,15.202518,17.004251,1.801733
23242,2024-11-04,1000,16,2.310619,2.584463,15.202518,17.004251,1.801733


# Cálculo de Costes de Stock

El coste diario de mantener stock se calcula como:

$$\text{Coste stock día} = \text{Porcentaje Coste Unitario} \times \text{Precio} \times \text{Unidades en stock}$$

Porcentaje Coste unitario = 5%

In [8]:
# Calcular valor del stock (precio * unidades)
df_test['valor_stock_CatBoost'] = df_test['stock_seguridad_CatBoost'] * df_test['eurPrecioMedio']
df_test['valor_stock_Naive'] = df_test['stock_seguridad_Naive'] * df_test['eurPrecioMedio']

# Calcular coste diario de almacenamiento
# Coste unitario asociado al stock = Coste de almacenaje + Coste de oportunidad de la inversión = %Coste unitario = 5%
tasa_diaria = TASA_ALMACENAMIENTO

df_test['coste_stock_dia_CatBoost'] = tasa_diaria * df_test['valor_stock_CatBoost']
df_test['coste_stock_dia_Naive'] = tasa_diaria * df_test['valor_stock_Naive']

# Calcular ahorro diario
df_test['ahorro_diario_euros'] = df_test['coste_stock_dia_Naive'] - df_test['coste_stock_dia_CatBoost']
df_test['ahorro_diario_%'] = np.where(
    df_test['coste_stock_dia_Naive'] == 0,
    0,
    (df_test['ahorro_diario_euros'] / df_test['coste_stock_dia_Naive']) * 100
)

# Estadísticas de costes
print("=" * 80)
print("COSTES DE STOCK - ESTADÍSTICAS DIARIAS")
print("=" * 80)
print(f"\nCoste promedio diario por producto según CatBoost: {df_test['coste_stock_dia_CatBoost'].mean():.2f} €")
print(f"Coste promedio diario por producto según Naive: {df_test['coste_stock_dia_Naive'].mean():.2f} €")
print(f"Ahorro promedio diario por producto: {df_test['ahorro_diario_euros'].mean():.2f} € ({df_test['ahorro_diario_%'].mean():.2f}%)")

# Mostrar ejemplos
print("\n" + "=" * 80)
print("EJEMPLOS DE COSTES DIARIOS")
print("=" * 80)
display(df_test[['fecha', 'producto', 'eurPrecioMedio', 
                  'stock_seguridad_CatBoost', 'stock_seguridad_Naive',
                  'coste_stock_dia_CatBoost', 'coste_stock_dia_Naive', 
                  'ahorro_diario_euros']])

COSTES DE STOCK - ESTADÍSTICAS DIARIAS

Coste promedio diario por producto según CatBoost: 37.91 €
Coste promedio diario por producto según Naive: 40.62 €
Ahorro promedio diario por producto: 2.71 € (4.59%)

EJEMPLOS DE COSTES DIARIOS


,fecha,producto,eurPrecioMedio,stock_seguridad_CatBoost,stock_seguridad_Naive,coste_stock_dia_CatBoost,coste_stock_dia_Naive,ahorro_diario_euros
0,2024-10-07,1,68.73,97.178791,108.930164,333.954916,374.338509,40.383593
1,2024-10-08,1,68.73,97.178791,108.930164,333.954916,374.338509,40.383593
2,2024-10-09,1,68.73,97.178791,108.930164,333.954916,374.338509,40.383593
3,2024-10-10,1,68.73,97.178791,108.930164,333.954916,374.338509,40.383593
4,2024-10-11,1,68.73,97.178791,108.930164,333.954916,374.338509,40.383593
...,...,...,...,...,...,...,...,...
23239,2024-10-31,1000,40.73,15.202518,17.004251,30.959929,34.629157,3.669228
23240,2024-11-01,1000,40.73,15.202518,17.004251,30.959929,34.629157,3.669228
23241,2024-11-02,1000,40.73,15.202518,17.004251,30.959929,34.629157,3.669228
23242,2024-11-04,1000,40.73,15.202518,17.004251,30.959929,34.629157,3.669228


# Análisis Agregado por Día

Calcular los costes por día para visualizar la evolución temporal.

In [9]:
# Agregar por fecha (día)
df_diario = df_test.groupby('fecha').agg({
    'coste_stock_dia_CatBoost': 'sum',
    'coste_stock_dia_Naive': 'sum',
    'ahorro_diario_euros': 'sum',
    'stock_seguridad_CatBoost': 'sum',
    'stock_seguridad_Naive': 'sum',
    'reduccion_stock_unidades': 'sum'
}).reset_index()

# Calcular acumulados
df_diario['coste_acumulado_CatBoost'] = df_diario['coste_stock_dia_CatBoost'].cumsum()
df_diario['coste_acumulado_Naive'] = df_diario['coste_stock_dia_Naive'].cumsum()
df_diario['ahorro_acumulado'] = df_diario['ahorro_diario_euros'].cumsum()

print("=" * 80)
print("RESUMEN DIARIO AGREGADO")
print("=" * 80)
print(f"Total días analizados (son 30 días menos los días que no está abierto): {len(df_diario)}")
print(f"Coste total CatBoost: {df_diario['coste_acumulado_CatBoost'].iloc[-1]:,.2f} €")
print(f"Coste total Naive: {df_diario['coste_acumulado_Naive'].iloc[-1]:,.2f} €")
print(f"\nAhorro acumulado final (CatBoost - Naive): {df_diario['ahorro_acumulado'].iloc[-1]:,.2f} €")

RESUMEN DIARIO AGREGADO
Total días analizados (son 30 días menos los días que no está abierto): 26
Coste total CatBoost: 881,212.93 €
Coste total Naive: 944,124.06 €

Ahorro acumulado final (CatBoost - Naive): 62,911.13 €


# Gráfica - Evolución de Costes Diarios

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Subplots: solo costes acumulados y ahorro acumulado
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Evolución de Costes Acumulados",
    "Ahorro Acumulado por Reducción de Stock"
))

# Costes acumulados
fig.add_trace(
    go.Scatter(
        x=df_diario['fecha'],
        y=df_diario['coste_acumulado_Naive'],
        mode='lines',
        name='Modelo Naive',
        line=dict(color='#e74c3c', width=2.5),
        hovertemplate='%{x|%Y-%m-%d}<br>Coste Naive: %{y:,.0f} EUR<extra></extra>'
    ),
    row=1, col=1
 )
fig.add_trace(
    go.Scatter(
        x=df_diario['fecha'],
        y=df_diario['coste_acumulado_CatBoost'],
        mode='lines',
        name='Modelo CatBoost',
        line=dict(color='#27ae60', width=2.5),
        hovertemplate='%{x|%Y-%m-%d}<br>Coste CatBoost: %{y:,.0f} EUR<extra></extra>'
    ),
    row=1, col=1
 )

# Ahorro acumulado
fig.add_trace(
    go.Scatter(
        x=df_diario['fecha'],
        y=df_diario['ahorro_acumulado'],
        mode='lines',
        name='Ahorro acumulado',
        line=dict(color='#3498db', width=3),
        fill='tozeroy',
        fillcolor='rgba(52, 152, 219, 0.35)',
        hovertemplate='%{x|%Y-%m-%d}<br>Ahorro acumulado: %{y:,.0f} EUR<extra></extra>'
    ),
    row=1, col=2
 )

# Layout
fig.update_layout(
    height=550, width=1200,
    showlegend=True,
    legend_title_text='',
    template='plotly_white'
 )

# Ejes
fig.update_xaxes(title_text='Fecha', tickformat='%Y-%m-%d', tickangle=45)
fig.update_yaxes(title_text='Coste Acumulado (EUR)', row=1, col=1, separatethousands=True)
fig.update_yaxes(title_text='Ahorro Acumulado (EUR)', row=1, col=2, separatethousands=True)

fig.show()

# Tabla Resumen Comparativa

In [11]:
# Calcular variables para la tabla resumen
coste_total_naive = df_diario['coste_acumulado_Naive'].iloc[-1]
coste_total_catboost = df_diario['coste_acumulado_CatBoost'].iloc[-1]
ahorro_total = df_diario['ahorro_acumulado'].iloc[-1]
dias_en_periodo = len(df_diario)
factor_anualizacion = 365 / dias_en_periodo

# Calcular reducción promedio de stock (filtrando valores válidos)
reduccion_stock_promedio = df_test[df_test['stock_seguridad_Naive'] > 0]['reduccion_stock_%'].mean()

# Crear tabla resumen final
resumen = pd.DataFrame({
    'Métrica': [
        'RMSE Promedio (unidades)',
        'Stock Seguridad Promedio (unidades)',
        'Valor Stock Promedio (€)',
        'Coste Diario Promedio (€)',
        'Coste Total Periodo (€)',
        'Ahorro Total Periodo (€)',
        'Ahorro Anual Proyectado (€)',
        'Reducción Stock (%)',
        'Reducción Costes (%)'
    ],
    'Modelo Naive': [
        f"{rmse_por_producto['RMSE_Naive'].mean():.2f}",
        f"{df_test['stock_seguridad_Naive'].mean():.2f}",
        f"{df_test['valor_stock_Naive'].mean():.2f}",
        f"{df_test['coste_stock_dia_Naive'].mean():.2f}",
        f"{coste_total_naive:,.2f}",
        '-',
        '-',
        '-',
        '-'
    ],
    'Modelo CatBoost': [
        f"{rmse_por_producto['RMSE_CatBoost'].mean():.2f}",
        f"{df_test['stock_seguridad_CatBoost'].mean():.2f}",
        f"{df_test['valor_stock_CatBoost'].mean():.2f}",
        f"{df_test['coste_stock_dia_CatBoost'].mean():.2f}",
        f"{coste_total_catboost:,.2f}",
        f"{ahorro_total:,.2f}",
        f"{ahorro_total * factor_anualizacion:,.2f}",
        f"{reduccion_stock_promedio:.2f}%",
        f"{(ahorro_total/coste_total_naive*100):.2f}%"
    ]
})

print("\n" + "=" * 100)
print("TABLA RESUMEN: COMPARATIVA MODELO NAIVE vs CATBOOST")
print("=" * 100)
display(resumen)

# Información adicional
print("\n" + "=" * 100)
print("PARÁMETROS UTILIZADOS")
print("=" * 100)
print(f"• Nivel de servicio: {NIVEL_SERVICIO*100}%")
print(f"• Factor Z (Z-score): {Z_SCORE:.3f}")
print(f"• Porcentaje a aplicar al coste unitario: {TASA_ALMACENAMIENTO*100}%")
print(f"• Periodo analizado: {df_test['fecha'].min().strftime('%Y-%m-%d')} a {df_test['fecha'].max().strftime('%Y-%m-%d')}")
print(f"• Días en el periodo: {dias_en_periodo}")
print(f"• Número de productos: {df_test['producto'].nunique()}")
print(f"• Total de observaciones: {len(df_test):,}")


TABLA RESUMEN: COMPARATIVA MODELO NAIVE vs CATBOOST


,Métrica,Modelo Naive,Modelo CatBoost
0,RMSE Promedio (unidades),2.14,2.03
1,Stock Seguridad Promedio (unidades),15.56,14.72
2,Valor Stock Promedio (€),812.36,758.23
3,Coste Diario Promedio (€),40.62,37.91
4,Coste Total Periodo (€),"944,124.06","881,212.93"
5,Ahorro Total Periodo (€),-,"62,911.13"
6,Ahorro Anual Proyectado (€),-,"883,175.45"
7,Reducción Stock (%),-,5.01%
8,Reducción Costes (%),-,6.66%



PARÁMETROS UTILIZADOS
• Nivel de servicio: 95.0%
• Factor Z (Z-score): 1.645
• Porcentaje a aplicar al coste unitario: 5.0%
• Periodo analizado: 2024-10-07 a 2024-11-05
• Días en el periodo: 26
• Número de productos: 894
• Total de observaciones: 23,244


# Análisis por Producto

In [13]:
# Agregar costes por producto
df_por_producto = df_test.groupby('producto').agg({
    'eurPrecioMedio': 'first',
    'ciclo_aprovisionamiento': 'first',
    'RMSE_CatBoost': 'first',
    'RMSE_Naive': 'first',
    'stock_seguridad_CatBoost': 'first',
    'stock_seguridad_Naive': 'first',
    'coste_stock_dia_CatBoost': 'sum',
    'coste_stock_dia_Naive': 'sum',
    'ahorro_diario_euros': 'sum'
}).reset_index()

# Calcular porcentajes con protección contra división por cero
df_por_producto['mejora_RMSE_%'] = np.where(
    df_por_producto['RMSE_Naive'] == 0,
    0,
    (df_por_producto['RMSE_Naive'] - df_por_producto['RMSE_CatBoost']) / df_por_producto['RMSE_Naive'] * 100
)

df_por_producto['reduccion_stock_%'] = np.where(
    df_por_producto['stock_seguridad_Naive'] == 0,
    0,
    (df_por_producto['stock_seguridad_Naive'] - df_por_producto['stock_seguridad_CatBoost']) / df_por_producto['stock_seguridad_Naive'] * 100
)

df_por_producto['ahorro_%'] = np.where(
    df_por_producto['coste_stock_dia_Naive'] == 0,
    0,
    df_por_producto['ahorro_diario_euros'] / df_por_producto['coste_stock_dia_Naive'] * 100
)

# Renombrar columnas para mejor visualización
df_por_producto.columns = [
    'Producto', 'Precio (€)', 'Ciclo Aprov. (días)', 
    'RMSE CatBoost', 'RMSE Naive',
    'Stock Segur. CatBoost', 'Stock Segur. Naive',
    'Coste Total CatBoost (€)', 'Coste Total Naive (€)', 
    'Ahorro Total (€)', 'Mejora RMSE (%)', 
    'Reducción Stock (%)', 'Ahorro (%)'
]

# Ordenar por ahorro
df_por_producto = df_por_producto.sort_values('Ahorro Total (€)', ascending=False)

print("\n" + "=" * 100)
print("ANÁLISIS DETALLADO POR PRODUCTO")
print("=" * 100)
display(df_por_producto)


ANÁLISIS DETALLADO POR PRODUCTO


,Producto,Precio (€),Ciclo Aprov. (días),RMSE CatBoost,RMSE Naive,Stock Segur. CatBoost,Stock Segur. Naive,Coste Total CatBoost (€),Coste Total Naive (€),Ahorro Total (€),Mejora RMSE (%),Reducción Stock (%),Ahorro (%)
505,572,625.260000,16,4.294293,4.801745,28.253935,31.592671,22965.872075,25679.723736,2713.851661,10.568072,10.568072,10.568072
2,3,169.000000,29,5.331253,6.600565,47.223213,58.466535,10374.939789,12845.097832,2470.158043,19.230356,19.230356,19.230356
76,85,189.996667,29,4.250378,5.107184,37.649030,45.238457,9299.147241,11173.702830,1874.555589,16.776494,16.776494,16.776494
127,144,97.040000,29,4.996389,6.614030,44.257052,58.585810,5583.115615,7390.717137,1807.601522,24.457728,24.457728,24.457728
260,289,323.500000,16,3.554628,4.058017,23.387373,26.699376,9835.559784,11228.422389,1392.862605,12.404793,12.404793,12.404793
...,...,...,...,...,...,...,...,...,...,...,...,...,...
388,438,177.462000,18,0.577515,0.000000,4.030205,0.000000,929.770725,0.000000,-929.770725,0.000000,0.000000,0.000000
738,835,189.000000,20,0.557590,0.000000,4.101635,0.000000,1007.771811,0.000000,-1007.771811,0.000000,0.000000,0.000000
728,820,257.470000,16,0.525250,0.000000,3.455840,0.000000,1156.707782,0.000000,-1156.707782,0.000000,0.000000,0.000000
242,270,387.350000,9,0.594800,0.000000,2.935076,0.000000,1477.972038,0.000000,-1477.972038,0.000000,0.000000,0.000000
